# ETL — switzerland Train Data

Ce notebook transforme les données GTFS autrichienne en un format standardisé avec les colonnes :
`data_source`, `route_id`, `id_origin_city`, `id_destination_city`, `weekly_train`, `desserte_type`

**Sources :** `data/switzerland/`
- `routes.csv` — infos sur les lignes
- `trips.csv` — association route ↔ trip
- `stop_times.csv` — séquence des arrêts par trip
- `stops.csv` — métadonnées des arrêts

## 0. Imports & configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../../../data/switzerland")
DATA_SOURCE = "switzerland"

## 1. Chargement des fichiers bruts

In [2]:
routes = pd.read_csv(
    DATA_DIR / "routes.csv",
    usecols=["route_id", "route_short_name", "route_long_name", "route_type"],
    dtype=str
)

trips = pd.read_csv(
    DATA_DIR / "trips.csv",
    usecols=["route_id", "trip_id"],
    dtype=str
)

stop_times = pd.read_csv(
    DATA_DIR / "stop_times.csv",
    usecols=["trip_id", "stop_id", "stop_sequence"],
    dtype={"trip_id": str, "stop_id": str, "stop_sequence": int}
)

stops = pd.read_csv(
    DATA_DIR / "stops.csv",
    usecols=["stop_id", "stop_name", "parent_station"],
    dtype=str
)

print(f"routes : {routes.shape}")
print(f"trips : {trips.shape}")
print(f"stop_times : {stop_times.shape}")
print(f"stops : {stops.shape}")

routes : (5060, 4)
trips : (1554411, 2)
stop_times : (24693166, 3)
stops : (102973, 3)


## 2. Extraction des arrêts origine / destination par trip

Pour chaque `trip_id`, on retient :
- **origin** = arrêt avec le `stop_sequence` le plus petit
- **destination** = arrêt avec le `stop_sequence` le plus grand

In [3]:
# Premier arrêt de chaque trip
origin = (
    stop_times
    .sort_values("stop_sequence")
    .groupby("trip_id", as_index=False)
    .first()[["trip_id", "stop_id"]]
    .rename(columns={"stop_id": "id_origin_city"})
)

# Dernier arrêt de chaque trip
destination = (
    stop_times
    .sort_values("stop_sequence")
    .groupby("trip_id", as_index=False)
    .last()[["trip_id", "stop_id"]]
    .rename(columns={"stop_id": "id_destination_city"})
)

od_per_trip = origin.merge(destination, on="trip_id")

# Filtrer les trips sans mouvement (origine == destination)
od_per_trip = od_per_trip[od_per_trip["id_origin_city"] != od_per_trip["id_destination_city"]]

print(f"Trips avec OD valides : {len(od_per_trip):,}")
od_per_trip.head()

Trips avec OD valides : 1,544,469


,trip_id,id_origin_city,id_destination_city
0,.ojp-91-1-A.1.TA.1.j26,8502204:0:5,8502206:0:2
1,.ojp-91-1-A.1.TA.10.j26,8502204:0:2,8502206:0:2
2,.ojp-91-1-A.1.TA.100.j26,8502007:0:2CD,8505000:0:7
3,.ojp-91-1-A.1.TA.1000.j26,8502007:0:2CD,8502206:0:2
4,.ojp-91-1-A.1.TA.1001.j26,8502007:0:2CD,8502206:0:2


## 3. Jointures : trips → routes + OD

In [4]:
# Associer chaque trip à sa route
trips_enriched = trips.merge(od_per_trip, on="trip_id", how="inner")

# Ajouter les infos de route
trips_enriched = trips_enriched.merge(routes, on="route_id", how="left")

print(f"Lignes après jointures : {len(trips_enriched):,}")
trips_enriched.head()

Lignes après jointures : 1,544,469


,route_id,trip_id,id_origin_city,id_destination_city,route_short_name,route_long_name,route_type
0,91-10-A-j26-1,.ojp-91-10-A.1.TA.1.j26,8503054:0:1,8503000:0:22,S10,NaN,109
1,91-10-A-j26-1,.ojp-91-10-A.1.TA.10.j26,8503054:0:1,8503000:0:22,S10,NaN,109
2,91-10-A-j26-1,.ojp-91-10-A.1.TA.100.j26,8503057:0:1,8503090:0:2,S10,NaN,109
3,91-10-A-j26-1,.ojp-91-10-A.1.TA.101.j26,8503057:0:1,8503090:0:2,S10,NaN,109
4,91-10-A-j26-1,.ojp-91-10-A.1.TA.102.j26,8503057:0:1,8503090:0:2,S10,NaN,109


## 4. Calcul de `weekly_train`

On agrège par `(route_id, id_origin_city, id_destination_city)` et on compte les trips distincts.
Ce comptage est un proxy du volume hebdomadaire (les données GTFS représentent typiquement une semaine type).

In [5]:
aggregated = (
    trips_enriched
    .groupby(["route_id", "id_origin_city", "id_destination_city"], as_index=False)
    .agg(weekly_train=("trip_id", "nunique"))
)

print(f"Paires OD uniques : {len(aggregated):,}")
print(f"\nDistribution weekly_train :")
print(aggregated["weekly_train"].describe())

Paires OD uniques : 47,364

Distribution weekly_train :
count    47364.000000
mean        32.608500
std        155.391359
min          1.000000
25%          1.000000
50%          4.000000
75%         15.000000
max       4869.000000
Name: weekly_train, dtype: float64


## 5. Calcul de `desserte_type`

| Condition | Valeur |
|---|---|
| `weekly_train < 7` | `Sous-desservi` |
| `7 ≤ weekly_train ≤ 56` | `Desserte Normale` |
| `weekly_train > 56` | `Bien desservi` |

In [6]:
def classify_desserte(n):
    if n < 7:
        return "Sous-desservi"
    elif n <= 56:
        return "Desserte Normale"
    else:
        return "Bien desservi"

aggregated["desserte_type"] = aggregated["weekly_train"].apply(classify_desserte)

print("Répartition desserte_type :")
print(aggregated["desserte_type"].value_counts())

Répartition desserte_type :
desserte_type
Sous-desservi       29156
Desserte Normale    14112
Bien desservi        4096
Name: count, dtype: int64


## 6. Construction du DataFrame final

In [7]:
OUTPUT_COLS = [
    "data_source",
    "route_id",
    "id_origin_city",
    "id_destination_city",
    "weekly_train",
    "desserte_type",
]

result = aggregated.copy()
result["data_source"] = DATA_SOURCE
result = result[OUTPUT_COLS].sort_values(["route_id", "id_origin_city", "id_destination_city"]).reset_index(drop=True)

print(f"Shape finale : {result.shape}")
print(f"Colonnes : {list(result.columns)}")
result.head(10)

Shape finale : (47364, 6)
Colonnes : ['data_source', 'route_id', 'id_origin_city', 'id_destination_city', 'weekly_train', 'desserte_type']


,data_source,route_id,id_origin_city,id_destination_city,weekly_train,desserte_type
0,switzerland,91-1-A-j26-1,8502007,8502204,1,Sous-desservi
1,switzerland,91-1-A-j26-1,8502007:0:1,8502020:0:1,1,Sous-desservi
2,switzerland,91-1-A-j26-1,8502007:0:1,8502021:0:1,6,Sous-desservi
3,switzerland,91-1-A-j26-1,8502007:0:1,8502021:0:2,3,Sous-desservi
4,switzerland,91-1-A-j26-1,8502007:0:1,8502204:0:4,3,Sous-desservi
5,switzerland,91-1-A-j26-1,8502007:0:1,8502206:0:2,16,Desserte Normale
6,switzerland,91-1-A-j26-1,8502007:0:1,8505000:0:10,2,Sous-desservi
7,switzerland,91-1-A-j26-1,8502007:0:1,8505000:0:11,3,Sous-desservi
8,switzerland,91-1-A-j26-1,8502007:0:1,8505000:0:2,7,Desserte Normale
9,switzerland,91-1-A-j26-1,8502007:0:1,8505000:0:3,2,Sous-desservi


## 7. Contrôles qualité

In [8]:
print("Valeurs nulles")
print(result.isnull().sum())

n_dup = result.duplicated(subset=["route_id", "id_origin_city", "id_destination_city"]).sum()
print(f"{n_dup} doublon(s) détecté(s)")

assert (result["weekly_train"] > 0).all(), "Des weekly_train nuls ou négatifs détectés !"

valid_types = {"Sous-desservi", "Desserte Normale", "Bien desservi"}
assert set(result["desserte_type"].unique()).issubset(valid_types)

Valeurs nulles
data_source            0
route_id               0
id_origin_city         0
id_destination_city    0
weekly_train           0
desserte_type          0
dtype: int64
0 doublon(s) détecté(s)


## 8. Export

In [9]:
OUTPUT_PATH = Path("../../../data/output/switzerland_etl.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

result.to_csv(OUTPUT_PATH, index=False)
print(f"Export {OUTPUT_PATH.resolve()}")
print(f"   {len(result):,} lignes exportées")

Export C:\Users\kevyn\Documents\Taff\B3\TPRE612\data\output\switzerland_etl.csv
   47,364 lignes exportées
